# US Inflation Forecasting Under Structural Change

**Supervisor-revised, reproducible analysis notebook**

This notebook compares US year-over-year CPI inflation forecasts using a random-walk benchmark, seasonal-naïve benchmark, SARIMAX, LightGBM and Chronos-2. It evaluates one-month-ahead forecast accuracy across predefined economic regimes and evaluates Chronos-2 at 1-, 3-, 6- and 12-month horizons, including its nominal 80% prediction intervals.

### Research questions
1. Do statistical, machine-learning and foundation-model approaches improve on the last-value random-walk benchmark?
2. Does relative model performance change across pre-COVID, COVID-disruption and post-COVID periods?
3. How does Chronos-2's point-forecast accuracy and interval calibration change at 3-, 6- and 12-month horizons?
4. Are observed forecast-accuracy differences statistically distinguishable using paired Diebold–Mariano tests?
5. What 12-month inflation path does Chronos-2 project from the latest valid target observation?

### Evaluation design
- Initial training period ends in **December 2017**.
- Evaluation begins in **January 2018** and runs through **September 2026**.
- SARIMAX and LightGBM are re-estimated **annually** on expanding training samples (through the previous December); their model specifications/hyperparameters remain fixed.
- Chronos-2 remains a pretrained zero-shot model (its weights are not fine-tuned) and receives an expanding past-only context at every forecast origin.
- Random Walk forecasts use the latest available YoY inflation observation before each forecast month.
- All one-step models are scored on the same valid target dates. October 2025 remains in the monthly timeline but is excluded from error calculations because its actual YoY inflation is missing.
- Multi-horizon backtesting is performed for Chronos-2 and Random Walk. Multi-step SARIMAX/LightGBM forecasts would require future paths for their exogenous predictors; actual future predictor observations are not supplied because that would leak information.

**Important interpretation:** the three regime labels are predefined descriptive calendar periods, not formal statistically estimated structural-break dates. The study evaluates forecast performance across those periods; it does not prove that a specific event caused a forecast error change.

## 0. Environment and package setup

Run the installation cell only if a required package is missing. Chronos-2 downloads pretrained weights from Hugging Face on first use, so internet access is required for that step.

In [ ]:
# If needed, uncomment and run this installation command, then restart the kernel.
# %pip install -q "chronos-forecasting>=2.0" "lightgbm>=4.0" statsmodels scikit-learn scipy seaborn pyarrow

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import warnings
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
TRAIN_END = pd.Timestamp("2017-12-01")
EVAL_START = pd.Timestamp("2018-01-01")
EVAL_END = pd.Timestamp("2026-09-01")
HORIZONS = [1, 3, 6, 12]

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## 2. Load and align the FRED data

Place `CPIAUCSL.csv`, `INDPRO.csv`, `UNRATE.csv` and `FEDFUNDS.csv` beside this notebook, or in `data/raw`. Files ending in `(1).csv` are also accepted. No source-series imputation is performed.

In [ ]:
SEARCH_DIRS = [Path("."), Path("data/raw"), Path("../data/raw"), Path("../data")]
SERIES_NAMES = ["CPIAUCSL", "INDPRO", "UNRATE", "FEDFUNDS"]


def locate_csv(series_name):
    for folder in SEARCH_DIRS:
        for filename in (f"{series_name}(1).csv", f"{series_name}.csv"):
            path = folder / filename
            if path.exists():
                return path
    raise FileNotFoundError(
        f"Cannot find {series_name}.csv. Put all four FRED CSVs beside the notebook or in data/raw."
    )

raw_series = {}
file_audit = []
for name in SERIES_NAMES:
    path = locate_csv(name)
    temp = pd.read_csv(path)
    if "observation_date" not in temp.columns or name not in temp.columns:
        raise ValueError(f"Unexpected columns in {path.name}: {temp.columns.tolist()}")
    temp["observation_date"] = pd.to_datetime(temp["observation_date"], errors="coerce")
    temp = temp.dropna(subset=["observation_date"]).set_index("observation_date").sort_index()
    if temp.index.has_duplicates:
        raise ValueError(f"Duplicate monthly dates found in {path.name}.")
    raw_series[name] = temp[name].astype(float)
    file_audit.append({
        "Series": name,
        "File": path.name,
        "Rows in file": len(temp),
        "First date": temp.index.min(),
        "Latest date": temp.index.max(),
        "Missing in file": int(temp[name].isna().sum())
    })

file_audit = pd.DataFrame(file_audit)
display(file_audit)

data = pd.concat(raw_series, axis=1).sort_index()
data = data.loc["1990-07-01":"2026-10-01"].asfreq("MS")
data.index.name = "timestamp"

print("Aligned data shape:", data.shape)
print("Aligned date range:", data.index.min().date(), "to", data.index.max().date())
print("Missing values after monthly alignment:\n", data.isna().sum())
display(data.tail(12))

assert data.index.is_monotonic_increasing
assert data.index.is_unique
assert len(data) == len(pd.date_range(data.index.min(), data.index.max(), freq="MS"))

## 3. Construct target and predictors

The target is monthly **YoY CPI inflation**. Industrial production is transformed to YoY growth. Unemployment and federal funds rates remain in their original percentage units. The CPI gap in October 2025 also makes October 2026 YoY inflation unavailable; September 2026 is therefore the latest valid target month.

In [ ]:
data["Inflation"] = (data["CPIAUCSL"] / data["CPIAUCSL"].shift(12) - 1.0) * 100.0
data["INDPRO_growth"] = (data["INDPRO"] / data["INDPRO"].shift(12) - 1.0) * 100.0

model_data = data[["Inflation", "INDPRO_growth", "UNRATE", "FEDFUNDS"]].copy()
actual = model_data["Inflation"].copy()

print("Missing values after transformations:\n", model_data.isna().sum())
print("Latest valid YoY inflation date:", actual.last_valid_index())
display(model_data.tail(15))

assert actual.last_valid_index() == pd.Timestamp("2026-09-01")

## 4. Exploratory analysis

The correlation matrix is descriptive and contemporaneous. It is not a causal analysis and is not the lag structure used by the forecasting models.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(model_data.index, model_data["Inflation"], label="YoY CPI inflation")
axes[0].set_title("US CPI Inflation (YoY)")
axes[0].set_ylabel("Inflation (%)")
axes[0].legend()
axes[1].plot(model_data.index, model_data["INDPRO_growth"], label="INDPRO YoY growth")
axes[1].plot(model_data.index, model_data["UNRATE"], label="Unemployment rate")
axes[1].plot(model_data.index, model_data["FEDFUNDS"], label="Federal funds rate")
axes[1].set_title("Macroeconomic predictors")
axes[1].legend(ncol=3)
plt.tight_layout()
plt.show()

corr_data = model_data.dropna()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_data.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Contemporaneous correlation matrix")
plt.tight_layout()
plt.show()

## 5. Define forecast origins and predictor construction

SARIMAX and LightGBM use one-month-lagged macroeconomic predictors. The single missing October 2025 UNRATE value makes the November 2025 one-month-lagged unemployment feature missing; for that one model input only, the latest previously available unemployment observation is carried forward by one month. The original UNRATE source series remains unchanged. LightGBM can also handle missing feature values natively.

In [ ]:
eval_dates = model_data.loc[EVAL_START:EVAL_END].index
if len(eval_dates) == 0:
    raise ValueError("No evaluation dates found. Check EVAL_START and EVAL_END.")

# One-month-lagged exogenous predictors for SARIMAX
arimax_exog = pd.DataFrame(index=model_data.index)
arimax_exog["INDPRO_growth_lag1"] = model_data["INDPRO_growth"].shift(1)
arimax_exog["UNRATE_lag1"] = model_data["UNRATE"].shift(1)
arimax_exog["FEDFUNDS_lag1"] = model_data["FEDFUNDS"].shift(1)
arimax_exog["UNRATE_lag1"] = arimax_exog["UNRATE_lag1"].ffill(limit=1)

# LightGBM features: every feature for month t uses values no later than t-1.
features = pd.DataFrame(index=model_data.index)
for lag in [1, 2, 3, 6, 12]:
    features[f"Inflation_lag{lag}"] = actual.shift(lag)
features["Inflation_rollmean3"] = actual.shift(1).rolling(3).mean()
features["Inflation_rollmean6"] = actual.shift(1).rolling(6).mean()
for col in ["INDPRO_growth", "UNRATE", "FEDFUNDS"]:
    features[f"{col}_lag1"] = model_data[col].shift(1)

# Initial SARIMAX training sample: all target/exog rows through Dec-2017.
sarimax_initial_mask = (model_data.index <= TRAIN_END) & arimax_exog.notna().all(axis=1)
y_sarimax_initial = actual.loc[sarimax_initial_mask]
X_sarimax_initial = arimax_exog.loc[sarimax_initial_mask]
MODEL_START = y_sarimax_initial.index.min()

if y_sarimax_initial.isna().any():
    raise ValueError("Unexpected missing target in initial SARIMAX training sample.")
if X_sarimax_initial.isna().any().any():
    raise ValueError("Missing exogenous data in the initial SARIMAX training sample.")

# Initial LightGBM training sample matches the feature-complete initial sample.
ml_initial = features.join(actual.rename("Target")).loc[:TRAIN_END].dropna()
if ml_initial.empty:
    raise ValueError("No complete initial LightGBM training rows.")
LGBM_TRAIN_START = ml_initial.index.min()

print("SARIMAX initial observations:", len(y_sarimax_initial))
print("SARIMAX training dates:", y_sarimax_initial.index.min().date(), "to", y_sarimax_initial.index.max().date())
print("LightGBM initial complete observations:", len(ml_initial))
print("LightGBM feature start:", LGBM_TRAIN_START.date())
print("Evaluation origins:", len(eval_dates), "from", eval_dates.min().date(), "to", eval_dates.max().date())

## 6. Random-walk and seasonal-naïve benchmarks

**Primary benchmark — Random Walk:** the next month's YoY inflation is forecast as the latest available YoY inflation observation. When a target month is missing, the benchmark carries forward the last actually observed target value; it never uses a future observation.

**Secondary benchmark — Seasonal naïve:** repeats the YoY inflation observation from 12 months earlier. It is retained for comparison with the earlier project, but is not treated as the primary benchmark for a seasonally adjusted YoY inflation rate.

In [ ]:
backtest = pd.DataFrame(index=eval_dates)
backtest.index.name = "timestamp"
backtest["Actual"] = actual.reindex(eval_dates)
backtest["Random Walk"] = actual.ffill().shift(1).reindex(eval_dates)
backtest["Seasonal Naive"] = actual.shift(12).reindex(eval_dates)

display(backtest.head())

## 7. SARIMAX specification selection using training data only

To avoid using the evaluation period when choosing the SARIMAX order, the candidate search below uses only the initial training sample ending December 2017. The ACF and PACF plots are also calculated from this initial training target only. The candidate set compares:
- the earlier seasonal AR specification, SARIMAX(1,0,1)(1,0,0)12;
- a non-seasonal ARIMAX(1,0,1) alternative;
- non-seasonal first-differenced ARIMAX(1,1,1);
- a first-differenced model with a seasonal AR term, SARIMAX(1,1,1)(1,0,0)12;
- a higher-order non-seasonal ARIMAX(2,0,1); and
- a seasonal MA alternative, SARIMAX(1,0,1)(0,0,1)12.

Here, `d=1` tests ordinary first differencing. **No seasonal differencing candidate (`D=1`) is included**; the search is deliberately small, and YoY inflation is already a 12-month rate rather than the unadjusted monthly CPI index. The seasonal period `s=12` tests whether a lag-12 AR or MA term improves the fit; it should not be interpreted as assuming that strong CPI seasonality remains in the YoY target.

Candidates are fitted with stationarity and invertibility constraints and compared using AIC/BIC on the same initial training observations and the same lagged exogenous predictors. The selection rule is the minimum BIC among candidates that converge and whose minimum AR and MA root moduli exceed 1.01. The theoretical stationarity/invertibility boundary is 1.0; the additional 1.01 margin is a conservative filter against near-boundary, weakly identified estimates. The selected candidate's residual diagnostics are then assessed separately. A low information criterion alone is not treated as proof of an adequate model.

The output table is the evidence for the final choice. Report the selected order, its AIC/BIC, the competing differenced/non-seasonal alternatives, and any candidate excluded by the convergence/root rule. A small BIC difference should be described as weak evidence rather than a decisive model-selection result.


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(y_sarimax_initial, lags=36, ax=axes[0], title="Initial-training ACF of YoY inflation")
plot_pacf(y_sarimax_initial, lags=36, method="ywm", ax=axes[1], title="Initial-training PACF of YoY inflation")
plt.tight_layout()
plt.show()

SARIMAX_CANDIDATES = [
    {"name": "SARIMAX(1,0,1)(1,0,0,12)", "order": (1, 0, 1), "seasonal_order": (1, 0, 0, 12)},
    {"name": "ARIMAX(1,0,1)", "order": (1, 0, 1), "seasonal_order": (0, 0, 0, 0)},
    {"name": "ARIMAX(1,1,1)", "order": (1, 1, 1), "seasonal_order": (0, 0, 0, 0)},
    {"name": "SARIMAX(1,1,1)(1,0,0,12)", "order": (1, 1, 1), "seasonal_order": (1, 0, 0, 12)},
    {"name": "ARIMAX(2,0,1)", "order": (2, 0, 1), "seasonal_order": (0, 0, 0, 0)},
    {"name": "SARIMAX(1,0,1)(0,0,1,12)", "order": (1, 0, 1), "seasonal_order": (0, 0, 1, 12)},
]

candidate_rows = []
candidate_fits = {}

for spec in SARIMAX_CANDIDATES:
    try:
        candidate_model = SARIMAX(
            y_sarimax_initial,
            exog=X_sarimax_initial,
            order=spec["order"],
            seasonal_order=spec["seasonal_order"],
            enforce_stationarity=True,
            enforce_invertibility=True
        )
        candidate_fit = candidate_model.fit(disp=False, maxiter=500)
        converged = bool(candidate_fit.mle_retvals.get("converged", False))
        ar_roots = np.abs(candidate_fit.arroots)
        ma_roots = np.abs(candidate_fit.maroots)
        # The theoretical condition is > 1.0. A 1.01 margin excludes numerically
        # near-boundary solutions that can have huge standard errors and weak identification.
        min_ar_root = float(ar_roots.min()) if len(ar_roots) else np.inf
        min_ma_root = float(ma_roots.min()) if len(ma_roots) else np.inf
        ar_ok = min_ar_root > 1.01
        ma_ok = min_ma_root > 1.01
        roots_ok = ar_ok and ma_ok
        candidate_fits[spec["name"]] = candidate_fit
        candidate_rows.append({
            "Model": spec["name"],
            "Order": spec["order"],
            "Seasonal order": spec["seasonal_order"],
            "AIC": candidate_fit.aic,
            "BIC": candidate_fit.bic,
            "Converged": converged,
            "Minimum AR root modulus": min_ar_root,
            "Minimum MA root modulus": min_ma_root,
            "AR roots stable (margin > 1.01)": ar_ok,
            "MA roots invertible (margin > 1.01)": ma_ok,
            "Eligible": converged and roots_ok
        })
    except Exception as exc:
        candidate_rows.append({
            "Model": spec["name"],
            "Order": spec["order"],
            "Seasonal order": spec["seasonal_order"],
            "AIC": np.nan,
            "BIC": np.nan,
            "Converged": False,
            "Minimum AR root modulus": np.nan,
            "Minimum MA root modulus": np.nan,
            "AR roots stable (margin > 1.01)": False,
            "MA roots invertible (margin > 1.01)": False,
            "Eligible": False,
            "Error": str(exc)[:180]
        })

sarimax_grid = pd.DataFrame(candidate_rows).sort_values("BIC", na_position="last").reset_index(drop=True)
display(sarimax_grid.round(4))

eligible = sarimax_grid.loc[sarimax_grid["Eligible"]].copy()
if eligible.empty:
    raise RuntimeError("No SARIMAX candidate both converged and passed the AR/MA root checks. Review the grid output.")

SELECTED_SARIMAX_NAME = eligible.iloc[0]["Model"]
SELECTED_SPEC = next(s for s in SARIMAX_CANDIDATES if s["name"] == SELECTED_SARIMAX_NAME)
SELECTED_ORDER = SELECTED_SPEC["order"]
SELECTED_SEASONAL_ORDER = SELECTED_SPEC["seasonal_order"]
SARIMAX_INITIAL_FIT = candidate_fits[SELECTED_SARIMAX_NAME]

print("Selected by lowest BIC among converged, root-stable candidates:", SELECTED_SARIMAX_NAME)
print("Order:", SELECTED_ORDER)
print("Seasonal order:", SELECTED_SEASONAL_ORDER)
print(SARIMAX_INITIAL_FIT.summary())

## 8. SARIMAX stationarity and residual diagnostics

ADF and KPSS are applied to the initial training target only. Residual tests are applied to the selected model fitted on the initial training sample. ADF and KPSS have opposite null hypotheses. Residual test failures are reported as model limitations; they do not automatically invalidate one-step point forecasts.

In [ ]:
train_inflation = y_sarimax_initial.dropna()
adf_stat, adf_p, *_ = adfuller(train_inflation, autolag="AIC")
kpss_stat, kpss_p, *_ = kpss(train_inflation, regression="c", nlags="auto")

residuals = pd.Series(SARIMAX_INITIAL_FIT.resid).dropna()
model_df = sum(SELECTED_ORDER[::2]) + SELECTED_SEASONAL_ORDER[0] + SELECTED_SEASONAL_ORDER[2]
model_df = int(model_df)
lb = acorr_ljungbox(residuals, lags=[12, 24], model_df=model_df, return_df=True)
jb_stat, jb_p, skewness, kurtosis = jarque_bera(residuals)
arch_stat, arch_p, _, _ = het_arch(residuals, nlags=12)
ar_roots = np.abs(SARIMAX_INITIAL_FIT.arroots)
ma_roots = np.abs(SARIMAX_INITIAL_FIT.maroots)

sarimax_diagnostics = pd.DataFrame([
    {"Diagnostic": "ADF (H0: unit root)", "Statistic": adf_stat, "P-value": adf_p},
    {"Diagnostic": "KPSS (H0: level stationarity)", "Statistic": kpss_stat, "P-value": kpss_p},
    {"Diagnostic": "Jarque-Bera (H0: normal residuals)", "Statistic": jb_stat, "P-value": jb_p},
    {"Diagnostic": "ARCH-LM, 12 lags (H0: no ARCH)", "Statistic": arch_stat, "P-value": arch_p},
])
display(sarimax_diagnostics.round(5))
print("Ljung-Box residual tests:")
display(lb.round(5))
print("Minimum AR root modulus:", float(ar_roots.min()) if len(ar_roots) else np.nan)
print("Minimum MA root modulus:", float(ma_roots.min()) if len(ma_roots) else np.nan)
print("AR roots outside unit circle:", bool(np.all(ar_roots > 1)) if len(ar_roots) else True)
print("MA roots outside unit circle:", bool(np.all(ma_roots > 1)) if len(ma_roots) else True)

SARIMAX_INITIAL_FIT.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

## 9. Expanding-window annual refits: SARIMAX and LightGBM

To address unequal update rules, the trainable models are re-estimated at the start of each calendar year using all eligible observations through the previous December. Within each year, SARIMAX updates its state monthly after the forecast is made; LightGBM uses newly available lagged features but retains that year's fitted estimator. Hyperparameters and the selected SARIMAX order remain fixed.

This is an **annual expanding-window refit strategy**, not a monthly refit experiment. Chronos-2 is handled separately as a pretrained zero-shot model with an expanding in-context history at each origin; its pretrained weights are not fine-tuned.

In [ ]:
# Fixed LightGBM configuration; estimator is re-fitted each January on an expanding sample.
LGBM_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1
)

sarimax_one_step = []
lgbm_one_step = []
sarimax_refit_log = []
lgbm_refit_log = []

sarimax_result = None
lgbm_model = None
fitted_year = None

for date in eval_dates:
    # Annual expanding-window refit: Jan forecast uses data through the prior December.
    if fitted_year != date.year:
        refit_cutoff = date - pd.offsets.MonthBegin(1)

        # SARIMAX: preserve the regular monthly index and genuine missing target values.
        sarimax_train_y = actual.loc[MODEL_START:refit_cutoff]
        sarimax_train_x = arimax_exog.loc[MODEL_START:refit_cutoff]
        if sarimax_train_x.isna().any().any():
            raise ValueError(f"Missing SARIMAX exog at annual refit ending {refit_cutoff.date()}.")

        sarimax_model = SARIMAX(
            sarimax_train_y,
            exog=sarimax_train_x,
            order=SELECTED_ORDER,
            seasonal_order=SELECTED_SEASONAL_ORDER,
            enforce_stationarity=True,
            enforce_invertibility=True
        )
        sarimax_result = sarimax_model.fit(disp=False, maxiter=500)
        sarimax_converged = bool(sarimax_result.mle_retvals.get("converged", False))
        if not sarimax_converged:
            # A second optimizer is attempted from the first solution.
            try:
                retry = sarimax_model.fit(
                    start_params=np.asarray(sarimax_result.params),
                    method="powell",
                    disp=False,
                    maxiter=500
                )
                if bool(retry.mle_retvals.get("converged", False)):
                    sarimax_result = retry
                    sarimax_converged = True
            except Exception:
                pass

        sarimax_refit_log.append({
            "Refit year": date.year,
            "Training through": refit_cutoff,
            "N endog timestamps": len(sarimax_train_y),
            "Observed targets": int(sarimax_train_y.notna().sum()),
            "Converged": sarimax_converged,
            "AIC": sarimax_result.aic,
            "BIC": sarimax_result.bic
        })
        if not sarimax_converged:
            print(f"Warning: SARIMAX did not report convergence for refit year {date.year}.")

        # LightGBM: annual expanding window. Missing feature values are supported by LightGBM;
        # missing target values are excluded from fitting.
        train_frame = features.join(actual.rename("Target")).loc[LGBM_TRAIN_START:refit_cutoff].copy()
        train_frame = train_frame.dropna(subset=["Target"])
        if train_frame.empty:
            raise ValueError(f"No LightGBM training rows at refit {date.year}.")

        lgbm_model = LGBMRegressor(**LGBM_PARAMS)
        lgbm_model.fit(train_frame[features.columns], train_frame["Target"])
        lgbm_refit_log.append({
            "Refit year": date.year,
            "Training through": refit_cutoff,
            "N training rows": len(train_frame),
            "Training target min": float(train_frame["Target"].min()),
            "Training target max": float(train_frame["Target"].max())
        })

        fitted_year = date.year

    # Forecast month t using only lagged exogenous variables available before t.
    x_now = arimax_exog.loc[[date]]
    sarimax_pred = float(sarimax_result.get_forecast(steps=1, exog=x_now).predicted_mean.iloc[0])
    sarimax_one_step.append(sarimax_pred)

    lgbm_x_now = features.loc[[date], features.columns]
    lgbm_pred = float(lgbm_model.predict(lgbm_x_now)[0])
    lgbm_one_step.append(lgbm_pred)

    # After forecasting, update SARIMAX state with the newly observed target.
    # A genuine missing target is appended as NaN; it is not interpolated.
    y_now = np.array([actual.loc[date]], dtype=float)
    sarimax_result = sarimax_result.extend(endog=y_now, exog=x_now.to_numpy())

backtest["SARIMAX"] = sarimax_one_step
backtest["LightGBM"] = lgbm_one_step

print("SARIMAX annual refits:")
display(pd.DataFrame(sarimax_refit_log))
print("LightGBM annual refits:")
display(pd.DataFrame(lgbm_refit_log))

# Compare each year's evaluation target range with the target range available
# to LightGBM at the start-of-year refit. This contextualizes extrapolation difficulty.
lgbm_range_rows = []
for log_row in lgbm_refit_log:
    year = log_row["Refit year"]
    year_dates = actual.loc[(actual.index.year == year) & (actual.index >= EVAL_START) & (actual.index <= EVAL_END)].dropna()
    if len(year_dates):
        lgbm_range_rows.append({
            "Evaluation year": year,
            "Training target min": log_row["Training target min"],
            "Training target max": log_row["Training target max"],
            "Evaluation target min": float(year_dates.min()),
            "Evaluation target max": float(year_dates.max()),
            "Months above training max": int((year_dates > log_row["Training target max"]).sum()),
            "Valid evaluation months": len(year_dates)
        })
lgbm_range_analysis = pd.DataFrame(lgbm_range_rows)
print("LightGBM training range versus same-year evaluation range:")
display(lgbm_range_analysis.round(3))

## 10. Load Chronos-2 and run the unified rolling-origin backtest

Chronos-2 is used without fine-tuning. At every monthly forecast origin it receives only the historical target and the three historical macroeconomic covariate sequences dated before the first forecast month. One 12-step prediction is generated per origin; its 1-, 3-, 6- and 12-month points/quantiles are retained. This supplies both the one-step forecast and the multi-horizon backtest from the same forecast calls.

In [ ]:
import torch
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Chronos-2 inference device:", DEVICE)
chronos_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=DEVICE
)
print("Chronos-2 loaded.")

CHRONOS_COLUMNS = ["Inflation", "INDPRO_growth", "UNRATE", "FEDFUNDS"]
chronos_history = model_data[CHRONOS_COLUMNS].copy()
chronos_records = []

for i, first_target_date in enumerate(eval_dates, start=1):
    # Strictly past-only context. The first forecast row corresponds to first_target_date.
    context = chronos_history.loc[chronos_history.index < first_target_date].copy()
    context.index.name = "timestamp"
    context_df = context.reset_index()
    context_df.insert(0, "item_id", "US_CPI")

    pred_df = chronos_pipeline.predict_df(
        context_df,
        prediction_length=12,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="item_id",
        timestamp_column="timestamp",
        target="Inflation",
        freq="MS"
    )
    pred_df = pred_df.sort_values("timestamp").reset_index(drop=True)
    if len(pred_df) < 12:
        raise RuntimeError(f"Chronos-2 returned only {len(pred_df)} steps at origin {first_target_date}.")

    for h in HORIZONS:
        target_date = first_target_date + pd.DateOffset(months=h - 1)
        if target_date > EVAL_END:
            continue

        pred_row = pred_df.iloc[h - 1]
        actual_value = actual.get(target_date, np.nan)

        # Random Walk: last actual inflation observation strictly before the first forecast month.
        prior_actuals = actual.loc[actual.index < first_target_date].dropna()
        rw_value = float(prior_actuals.iloc[-1]) if len(prior_actuals) else np.nan

        chronos_records.append({
            "Forecast_Start": first_target_date,
            "Target_Date": target_date,
            "Horizon": h,
            "Actual": float(actual_value) if pd.notna(actual_value) else np.nan,
            "Chronos_Forecast": float(pred_row["predictions"]),
            "Lower_80": float(pred_row["0.1"]),
            "Median_50": float(pred_row["0.5"]),
            "Upper_80": float(pred_row["0.9"]),
            "RandomWalk_Forecast": rw_value
        })

    if i % 12 == 0 or i == len(eval_dates):
        print(f"Completed Chronos-2 forecast origin {i}/{len(eval_dates)}: {first_target_date:%Y-%m}")

chronos_multi_backtest = pd.DataFrame(chronos_records)

# One-step Chronos-2 forecast extracted from horizon 1.
chronos_h1 = chronos_multi_backtest.loc[chronos_multi_backtest["Horizon"] == 1].set_index("Target_Date")
backtest["Chronos-2"] = chronos_h1["Chronos_Forecast"].reindex(backtest.index)
backtest["Chronos-2_Lower80"] = chronos_h1["Lower_80"].reindex(backtest.index)
backtest["Chronos-2_Upper80"] = chronos_h1["Upper_80"].reindex(backtest.index)

print("Chronos-2 rolling-origin backtest complete.")
display(chronos_multi_backtest.head(8))

## 11. One-month-ahead forecast accuracy: common dates and random-walk-relative errors

All models are compared on the intersection of dates where the actual target and every forecast are available. Random Walk is the primary benchmark; seasonal naïve remains a secondary benchmark.

In [ ]:
MODEL_COLUMNS = ["Random Walk", "Chronos-2", "SARIMAX", "LightGBM", "Seasonal Naive"]
required = ["Actual"] + MODEL_COLUMNS
missing_cols = [c for c in required if c not in backtest.columns]
if missing_cols:
    raise ValueError(f"Backtest missing columns: {missing_cols}")

common_backtest = backtest[required].dropna()
print("Common one-step observations:", len(common_backtest))
print("Common evaluation dates:", common_backtest.index.min(), "to", common_backtest.index.max())

metric_rows = []
for model_name in MODEL_COLUMNS:
    y_true = common_backtest["Actual"]
    y_pred = common_backtest[model_name]
    metric_rows.append({
        "Model": model_name,
        "N": len(common_backtest),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    })

overall_metrics = pd.DataFrame(metric_rows).sort_values("RMSE").reset_index(drop=True)
rw_rmse = float(overall_metrics.loc[overall_metrics["Model"] == "Random Walk", "RMSE"].iloc[0])
overall_metrics["RMSE vs Random Walk (%)"] = (overall_metrics["RMSE"] / rw_rmse - 1) * 100
display(overall_metrics.round(4))

assert common_backtest.index.equals(backtest.loc[common_backtest.index].index)
assert len(common_backtest) == int(backtest["Actual"].notna().sum())

## 12. Forecast accuracy across predefined economic regimes

Regimes are predefined calendar partitions: pre-COVID (2018–2019), COVID disruption (2020–2021) and post-COVID (2022–September 2026). They are not formally estimated break dates. A common valid-date set is used within each regime across all models.

In [ ]:
def assign_regime(date):
    if date < pd.Timestamp("2020-01-01"):
        return "Pre-COVID (2018-2019)"
    if date < pd.Timestamp("2022-01-01"):
        return "COVID disruption (2020-2021)"
    return "Post-COVID (2022-Sep 2026)"

backtest["Regime"] = [assign_regime(d) for d in backtest.index]
regime_rows = []
for regime, group in backtest.groupby("Regime", sort=False):
    valid = group[["Actual"] + MODEL_COLUMNS].dropna()
    for model_name in MODEL_COLUMNS:
        regime_rows.append({
            "Regime": regime,
            "Model": model_name,
            "N": len(valid),
            "RMSE": np.sqrt(mean_squared_error(valid["Actual"], valid[model_name])),
            "MAE": mean_absolute_error(valid["Actual"], valid[model_name])
        })
regime_metrics = pd.DataFrame(regime_rows)
regime_rw = regime_metrics.loc[regime_metrics["Model"] == "Random Walk", ["Regime", "RMSE"]].rename(columns={"RMSE": "Random Walk RMSE"})
regime_metrics = regime_metrics.merge(regime_rw, on="Regime", how="left")
regime_metrics["RMSE vs Random Walk (%)"] = (regime_metrics["RMSE"] / regime_metrics["Random Walk RMSE"] - 1) * 100
display(regime_metrics.round(4))

## 13. Diebold–Mariano tests with HAC/Newey–West variance

The null hypothesis is equal expected predictive loss. The loss differential is defined as **Model A squared error minus Model B squared error**; a positive mean differential means Model B has lower average squared error. A Bartlett-kernel Newey–West long-run variance is used. For multi-step forecasts, the bandwidth is at least `horizon - 1` to account for overlapping forecast errors. The Harvey–Leybourne–Newbold finite-sample correction is applied. Holm-adjusted p-values are also reported for each family of comparisons.

These are paired forecast comparisons. Failure to reject the null is not proof that two models are identical; it means the sample does not provide sufficient evidence of a difference.

In [ ]:
def dm_hac_test(actual_values, forecast_a, forecast_b, horizon=1):
    """HLN-corrected Diebold-Mariano test for squared-error loss with Bartlett HAC variance."""
    frame = pd.concat([
        pd.Series(actual_values, name="actual"),
        pd.Series(forecast_a, name="forecast_a"),
        pd.Series(forecast_b, name="forecast_b")
    ], axis=1).dropna()
    n = len(frame)
    if n < max(20, horizon + 2):
        return {"N": n, "DM statistic": np.nan, "P-value": np.nan, "Mean loss diff A-B": np.nan, "HAC lags": np.nan}

    e_a = frame["actual"].to_numpy() - frame["forecast_a"].to_numpy()
    e_b = frame["actual"].to_numpy() - frame["forecast_b"].to_numpy()
    d = e_a**2 - e_b**2
    d_bar = float(np.mean(d))
    u = d - d_bar

    # Newey-West automatic bandwidth, increased to at least h-1 for overlapping h-step errors.
    nw_auto = int(np.floor(4 * (n / 100.0) ** (2 / 9)))
    bandwidth = min(n - 1, max(horizon - 1, nw_auto))

    long_run_var = float(np.dot(u, u) / n)
    for lag in range(1, bandwidth + 1):
        gamma = float(np.dot(u[lag:], u[:-lag]) / n)
        weight = 1.0 - lag / (bandwidth + 1.0)
        long_run_var += 2.0 * weight * gamma

    long_run_var = max(long_run_var, 1e-12)
    dm_stat = d_bar / np.sqrt(long_run_var / n)

    # Harvey-Leybourne-Newbold correction.
    hln_sq = (n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n
    if hln_sq > 0:
        dm_stat *= np.sqrt(hln_sq)
    p_value = float(2 * student_t.sf(abs(dm_stat), df=n - 1))

    return {
        "N": n,
        "DM statistic": float(dm_stat),
        "P-value": p_value,
        "Mean loss diff A-B": d_bar,
        "HAC lags": bandwidth
    }

# Primary one-step comparisons.
one_step_pairs = [
    ("Random Walk", "Chronos-2"),
    ("Random Walk", "SARIMAX"),
    ("Random Walk", "LightGBM"),
    ("Chronos-2", "SARIMAX")
]
one_step_dm_rows = []
for model_a, model_b in one_step_pairs:
    result = dm_hac_test(common_backtest["Actual"], common_backtest[model_a], common_backtest[model_b], horizon=1)
    one_step_dm_rows.append({"Comparison": f"{model_a} vs {model_b}", **result})
one_step_dm = pd.DataFrame(one_step_dm_rows)
one_step_dm["Holm adjusted p"] = multipletests(one_step_dm["P-value"].fillna(1), method="holm")[1]
display(one_step_dm.round(5))

# Multi-horizon paired comparisons: Chronos-2 against Random Walk.
multih_dm_rows = []
for h in [3, 6, 12]:
    sub = chronos_multi_backtest.loc[chronos_multi_backtest["Horizon"] == h].dropna(
        subset=["Actual", "Chronos_Forecast", "RandomWalk_Forecast"]
    )
    result = dm_hac_test(sub["Actual"], sub["Chronos_Forecast"], sub["RandomWalk_Forecast"], horizon=h)
    multih_dm_rows.append({"Horizon": h, "Comparison": "Chronos-2 vs Random Walk", **result})
multih_dm = pd.DataFrame(multih_dm_rows)
multih_dm["Holm adjusted p"] = multipletests(multih_dm["P-value"].fillna(1), method="holm")[1]
display(multih_dm.round(5))

## 14. Multi-horizon backtest and 80% prediction-interval evaluation

For each historical forecast origin, Chronos-2 generates a 12-step path. The 1-, 3-, 6- and 12-month forecasts are compared with a Random Walk that holds the latest available inflation observation constant across the horizon. This comparison uses the same valid origin–target pairs for both point forecasts.

The central 80% interval is defined by the 10th and 90th percentiles. We report empirical coverage, a Wilson confidence interval for coverage, mean interval width and the 80% interval score. Coverage close to 80% is desirable, while a lower interval score is preferable. Interval coverage alone should not be interpreted without interval width.

SARIMAX is not included in this multi-step comparison because its forecasts require future exogenous predictor values. We do not supply realized future predictors, as doing so would leak information. A separate model for forecasting the predictor paths would be needed for a valid multi-step SARIMAX comparison.

In [ ]:
multih_rows = []
for h in HORIZONS:
    sub = chronos_multi_backtest.loc[chronos_multi_backtest["Horizon"] == h].dropna(
        subset=["Actual", "Chronos_Forecast", "RandomWalk_Forecast", "Lower_80", "Upper_80"]
    ).copy()
    if sub.empty:
        continue

    y = sub["Actual"]
    cp = sub["Chronos_Forecast"]
    rw = sub["RandomWalk_Forecast"]
    inside = ((y >= sub["Lower_80"]) & (y <= sub["Upper_80"]))
    covered = int(inside.sum())
    n = len(sub)
    cov_lo, cov_hi = proportion_confint(covered, n, alpha=0.05, method="wilson")

    alpha = 0.20
    interval_score = (
        sub["Upper_80"] - sub["Lower_80"]
        + (2 / alpha) * (sub["Lower_80"] - y).clip(lower=0)
        + (2 / alpha) * (y - sub["Upper_80"]).clip(lower=0)
    ).mean()

    multih_rows.append({
        "Horizon (months)": h,
        "N": n,
        "Chronos RMSE": np.sqrt(mean_squared_error(y, cp)),
        "Random Walk RMSE": np.sqrt(mean_squared_error(y, rw)),
        "Chronos MAE": mean_absolute_error(y, cp),
        "Random Walk MAE": mean_absolute_error(y, rw),
        "Chronos RMSE vs RW (%)": (np.sqrt(mean_squared_error(y, cp)) / np.sqrt(mean_squared_error(y, rw)) - 1) * 100,
        "80% coverage (%)": 100 * covered / n,
        "Coverage 95% CI low (%)": 100 * cov_lo,
        "Coverage 95% CI high (%)": 100 * cov_hi,
        "Mean interval width": (sub["Upper_80"] - sub["Lower_80"]).mean(),
        "80% interval score": interval_score
    })

multi_horizon_metrics = pd.DataFrame(multih_rows)
display(multi_horizon_metrics.round(4))

# Check that the horizon-1 backtest reproduces the one-step Chronos-2 score on common dates.
h1 = chronos_multi_backtest.loc[chronos_multi_backtest["Horizon"] == 1].dropna(subset=["Actual", "Chronos_Forecast"])
h1_rmse = np.sqrt(mean_squared_error(h1["Actual"], h1["Chronos_Forecast"]))
print("Horizon-1 Chronos RMSE:", round(h1_rmse, 4))

## 15. Plot one-step forecasts and multi-horizon results

In [ ]:
plt.figure(figsize=(15, 7))
plt.plot(backtest.index, backtest["Actual"], label="Actual", color="black", linewidth=1.8)
for col in MODEL_COLUMNS:
    plt.plot(backtest.index, backtest[col], label=col, alpha=0.8)
plt.axvline(pd.Timestamp("2020-01-01"), linestyle="--", color="grey", label="COVID period begins")
plt.axvline(pd.Timestamp("2022-01-01"), linestyle=":", color="grey", label="Post-COVID period begins")
plt.title("Rolling one-month-ahead inflation forecasts")
plt.ylabel("YoY CPI inflation (%)")
plt.xlabel("Date")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

if not multi_horizon_metrics.empty:
    plt.figure(figsize=(9, 5))
    plt.plot(multi_horizon_metrics["Horizon (months)"], multi_horizon_metrics["Chronos RMSE"], marker="o", label="Chronos-2")
    plt.plot(multi_horizon_metrics["Horizon (months)"], multi_horizon_metrics["Random Walk RMSE"], marker="o", label="Random Walk")
    plt.xlabel("Forecast horizon (months)")
    plt.ylabel("RMSE (percentage points)")
    plt.title("Multi-horizon point-forecast accuracy")
    plt.xticks(HORIZONS)
    plt.legend()
    plt.tight_layout()
    plt.show()

backtest.to_csv("US_Inflation_OneStep_Backtest_Final.csv")
chronos_multi_backtest.to_csv("US_Inflation_MultiHorizon_Backtest_Final.csv", index=False)
overall_metrics.to_csv("US_Inflation_Overall_Metrics_Final.csv", index=False)
regime_metrics.to_csv("US_Inflation_Regime_Metrics_Final.csv", index=False)
multi_horizon_metrics.to_csv("US_Inflation_MultiHorizon_Metrics_Final.csv", index=False)

## 16. Final 12-month Chronos-2 forecast

The latest valid YoY inflation observation is September 2026. October 2026 YoY inflation is unavailable because October 2025 CPI is missing. The final projection therefore begins in October 2026 and ends in September 2027. The context is restricted through September 2026; no future macroeconomic predictor values are supplied.

A Random Walk reference forecast is also included. It holds the latest valid inflation observation constant over the 12-month horizon. This is a benchmark path, not a second multivariate forecast.

In [ ]:
last_valid_target_date = actual.last_valid_index()
assert last_valid_target_date == pd.Timestamp("2026-09-01")

future_context = chronos_history.loc[:last_valid_target_date].copy()
future_context.index.name = "timestamp"
future_context_df = future_context.reset_index()
future_context_df.insert(0, "item_id", "US_CPI")

future_all = chronos_pipeline.predict_df(
    future_context_df,
    prediction_length=12,
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id",
    timestamp_column="timestamp",
    target="Inflation",
    freq="MS"
)
future_all = future_all.sort_values("timestamp").reset_index(drop=True)

future_forecast = future_all[["timestamp", "predictions", "0.1", "0.5", "0.9"]].rename(columns={
    "timestamp": "Date",
    "predictions": "Chronos_Forecast",
    "0.1": "Lower_80",
    "0.5": "Median_50",
    "0.9": "Upper_80"
})

last_inflation = float(actual.loc[last_valid_target_date])
future_forecast["Random_Walk_Reference"] = last_inflation

display(future_forecast.round(3))
future_forecast.to_csv("US_CPI_Inflation_Forecast_Oct2026_Sep2027_Final.csv", index=False)
print("Saved 12-month forecast CSV.")

In [ ]:
# Verify forecast origin, month sequence, interval ordering and no future covariates.
expected_future_dates = pd.date_range("2026-10-01", periods=12, freq="MS")
assert future_context.index.max() == pd.Timestamp("2026-09-01")
assert future_forecast["Date"].reset_index(drop=True).equals(pd.Series(expected_future_dates, name="Date"))
assert len(future_forecast) == 12
assert (future_forecast["Lower_80"] <= future_forecast["Upper_80"]).all()
assert future_context.index.max() <= last_valid_target_date

print("Latest valid target:", last_valid_target_date.date())
print("Historical context ends:", future_context.index.max().date())
print("Forecast period:", future_forecast["Date"].min().date(), "to", future_forecast["Date"].max().date())
print("Forecast months:", len(future_forecast))
print("Forecast origin checks passed.")

In [ ]:
recent_history = actual.loc["2022-01-01":"2026-09-01"]
plt.figure(figsize=(13, 6))
plt.plot(recent_history.index, recent_history.values, label="Historical YoY inflation", color="black")
plt.plot(future_forecast["Date"], future_forecast["Chronos_Forecast"], marker="o", label="Chronos-2 forecast")
plt.plot(future_forecast["Date"], future_forecast["Random_Walk_Reference"], linestyle="--", label="Random Walk reference")
plt.fill_between(
    future_forecast["Date"],
    future_forecast["Lower_80"],
    future_forecast["Upper_80"],
    alpha=0.2,
    label="Nominal 80% prediction interval"
)
plt.axvline(pd.Timestamp("2026-10-01"), linestyle="--", color="grey", label="Forecast begins")
plt.title("US CPI inflation forecast: October 2026–September 2027")
plt.ylabel("YoY CPI inflation (%)")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

## 17. Interpretation and reporting checklist

After running the notebook, report the actual outputs from the metric tables. Do not copy earlier results from previous notebook versions: annual expanding-window refits and training-only SARIMAX selection change the experiment and can change the scores.

### What to report
1. Overall one-step RMSE and MAE for Random Walk, Chronos-2, SARIMAX, LightGBM and seasonal naïve, all on the same valid dates.
2. RMSE relative to Random Walk, where 0% means equal RMSE, positive values mean worse than Random Walk, and negative values mean better.
3. Regime-wise RMSE/MAE, with sample sizes.
4. Diebold–Mariano results, including HAC lag and Holm-adjusted p-values. A non-significant p-value is not evidence of identical models.
5. Chronos-2 3-, 6- and 12-month RMSE/MAE against Random Walk, plus empirical 80% coverage, coverage confidence interval, interval width and interval score.
6. SARIMAX training-only candidate grid, selected order, convergence/root checks and residual diagnostics.
7. Final Chronos-2 path and Random Walk reference path for October 2026–September 2027.

### Limitations to state
- The regime boundaries are descriptive calendar periods; no formal structural-break test estimates break dates.
- SARIMAX and LightGBM are re-estimated annually on expanding samples. Chronos-2 is not fine-tuned; its historical context expands each month in zero-shot mode. Random Walk and seasonal naïve update from observed inflation.
- The one-month-lagged predictors are used as a practical availability proxy. Actual publication delays, real-time data vintages and FRED revisions are not explicitly reconstructed.
- LightGBM is a tree ensemble and may have limited ability to extrapolate beyond target and feature patterns represented in its training data; interpret poor performance as sample-specific, not a universal property of the algorithm.
- SARIMAX order selection is limited to the small candidate grid shown and the initial training sample. Residual diagnostics may still reveal misspecification.
- Multi-horizon point and interval backtesting is performed for Chronos-2 and Random Walk. A multi-step SARIMAX forecast requires future exogenous predictor paths; no realized future predictors are used. Forecasting those predictor paths is outside this project.
- Chronos-2 pretraining data include subsets of Chronos Datasets, GIFT-Eval pretraining data and synthetic data. The model card does not identify whether these exact FRED series were included; potential pretraining overlap cannot be ruled out.
- Forecast-accuracy tests have limited power in a sample of 104 one-step pairs and fewer multi-horizon pairs. Report p-values and sample sizes, not just rankings.

### Literature and data references
- Atkeson, A., & Ohanian, L. E. (2001). Are Phillips curves useful for forecasting inflation? *Federal Reserve Bank of Minneapolis Quarterly Review*, 25(1), 2–11. https://www.minneapolisfed.org/research/qr/qr2511.html
- Stock, J. H., & Watson, M. W. (2007). Why has U.S. inflation become harder to forecast? *Journal of Money, Credit and Banking*, 39(s1), 3–33. https://doi.org/10.1111/j.1538-4616.2007.00014.x
- Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263. https://doi.org/10.1080/07350015.1995.10524599
- Ansari, A. F., et al. (2025). Chronos-2: From univariate to universal forecasting. arXiv:2510.15821. https://arxiv.org/abs/2510.15821
- Amazon. Chronos-2 model card and training-data description. https://huggingface.co/amazon/chronos-2
- Federal Reserve Bank of St. Louis. FRED series CPIAUCSL, INDPRO, UNRATE and FEDFUNDS. https://fred.stlouisfed.org/
- Hyndman, R. J., & Athanasopoulos, G. *Forecasting: Principles and Practice* (3rd ed.). OTexts. https://otexts.com/fpp3/

The written report should accompany this notebook. The report presents the objectives, methodology, findings and limitations; this notebook is the reproducibility supplement.

## 18. Final execution checks and export

Run the notebook from a fresh kernel using **Restart Kernel → Run All**. The final checks below verify that the main evaluation objects exist and that the same one-step evaluation sample is used across models. The notebook deliberately contains no pre-saved outputs; all tables and plots are generated by the current run.

In [ ]:
# Final reproducibility checks
assert set(MODEL_COLUMNS).issubset(backtest.columns)
assert len(common_backtest) == len(backtest[["Actual"] + MODEL_COLUMNS].dropna())
assert set(HORIZONS).issubset(set(chronos_multi_backtest["Horizon"].unique()))
assert future_forecast["Date"].min() == pd.Timestamp("2026-10-01")
assert future_forecast["Date"].max() == pd.Timestamp("2027-09-01")
assert len(future_forecast) == 12

print("Final notebook checks passed.")
print("One-step common observations:", len(common_backtest))
print("One-step evaluation period:", common_backtest.index.min().date(), "to", common_backtest.index.max().date())
print("Selected SARIMAX:", SELECTED_SARIMAX_NAME)
print("Multi-horizon sample sizes:")
display(multi_horizon_metrics[["Horizon (months)", "N", "Chronos RMSE", "Random Walk RMSE", "80% coverage (%)"]])
print("Files written: one-step backtest, multi-horizon backtest, overall metrics, regime metrics, multi-horizon metrics and 12-month forecast.")